# Evaluación en el Test Gold final (#104 / A4)

Evalúa contra el **test gold v2 final** (`test_gold_final_anotar/test_gold_final.csv`, 1533 citas) los clasificadores entrenados en
los notebooks 03–06: #104-b v2 (06), ganadores de #104 (04/05) + TF-IDF y A4 (03). No entrena nada; solo infiere y mide.

- **Gold**: `concordancia` (186), `una_anotacion` (820), `discrepancia_opus` (57, manda Opus) y `solo_opus` (470).
  Se reportan métricas globales (`todo`), del subconjunto con humano (`humano`, sin `solo_opus`) y del de cobertura B (`cob`, 839).
- **Sin QLoRA / jurado / LLMs.**
- `DRY = True` recorta a 64 casos, usa una BD de MLflow aparte y escribe en carpetas `*_dry`.

In [1]:
import gc
import json
import subprocess
import sys
import time
import traceback
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "packages/clasificacion-funcion/src"))
sys.path.insert(0, str(REPO / "packages/auditoria-corpus/src"))
sys.path.insert(0, str(REPO / "packages/recuperacion-densa/src"))
sys.path.insert(0, str(REPO))

import mlflow
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from clasificacion_funcion import adaptador_proyecto_grado as ad
from clasificacion_funcion import clasificador_torch as ct
from clasificacion_funcion import cli, dominio
from clasificacion_funcion import entrada_b_104 as eb
from clasificacion_funcion import experimento_104 as ex
from clasificacion_funcion import registro_mlflow as rm
from clasificacion_funcion import seleccion_datos as sd
from clasificacion_funcion import servicio

DRY = False   # True: 64 casos, BD de MLflow y carpetas aparte (*_dry)
LIM_DRY = 64
SUFIJO = "_dry" if DRY else ""
N_BOOTSTRAP = 200 if DRY else 5000
SEMILLA_BOOTSTRAP = 2026

ART = REPO / "proyecto_de_grado/artifacts"
EVIDENCIAS = REPO / f"proyecto_de_grado/docs/evidencias_test_gold{SUFIJO}"
PRED_DIR = ART / f"test_gold{SUFIJO}"
EV_B = REPO / "proyecto_de_grado/docs/evidencias_experimentos_104_b_v2"
CORRIDAS_104 = ART / "experimentos_104/corridas"
CORRIDAS_A4 = ART / "clasificacion_a4/corridas"
GOLD_CSV = REPO / "test_gold_final_anotar/test_gold_final.csv"
EVIDENCIAS.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)
FALLAS = []
T0 = time.time()


def liberar_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def escribir_json(nombre: str, obj) -> Path:
    ruta = EVIDENCIAS / nombre
    ruta.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    return ruta

/home/carlos/microproyecto-local-citation/local_workflows/.venv-qlora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Gold

Se regenera `test_gold_final.csv` con `scripts/consolidar_test_gold.py` (valida 1533 ids únicos = `ids_gold_v2()`).
`GOLD = {context_id: índice}` solo con casos evaluables (las abstenciones se predicen pero no puntúan).

In [2]:
subprocess.run([sys.executable, str(REPO / "scripts/consolidar_test_gold.py")], check=True, capture_output=True)
GOLD_SHA = sd.sha_archivo(GOLD_CSV)
GOLD_DF = pd.read_csv(GOLD_CSV, encoding="utf-8-sig")
assert len(GOLD_DF) == 1533 and GOLD_DF["context_id"].is_unique

IDS_TODO = sorted(GOLD_DF["context_id"])
IDS = IDS_TODO[:LIM_DRY] if DRY else IDS_TODO          # ids sobre los que se infiere
GOLD_USO = GOLD_DF[GOLD_DF["context_id"].isin(IDS)]
GOLD = {r.context_id: int(r.indice) for r in GOLD_USO.itertuples() if r.evaluable}
FUENTE = dict(zip(GOLD_USO["context_id"], GOLD_USO["fuente"]))
HUMANO = {c for c in GOLD if FUENTE[c] != "solo_opus"}
print("gold_sha256:", GOLD_SHA, "| ids:", len(IDS), "| evaluables:", len(GOLD), "| humano:", len(HUMANO))
print(pd.crosstab(GOLD_USO["fuente"], GOLD_USO["etiqueta"]).to_string())

gold_sha256: 80e4d9a6a72bd36913548bef897ad8212178797e181bb7893f7be2aad66b83c7 | ids: 1533 | evaluables: 1531 | humano: 1061
etiqueta           Application  Background  Basis  Comparison  Evidence  Further Reading  Gap  Identification of the Originator  Improvement / Modification  No evaluable
fuente                                                                                                                                                                   
concordancia                31          48      4          36        30               11   15                                 4                           7             0
discrepancia_opus           13          23      6           4         4                0    6                                 0                           1             0
solo_opus                  110         174     24          72        52                5   20                                 8                           5             0
una_anotacion             

## 2. Fuentes, entrada B y control de fugas

La entrada B del Gold se arma como en el notebook 06 (misma intersección de cobertura sobre `gold_v2`), con el Top-3 ya calculado
(`top3_gold_v2.csv` de #104-b v2). `COB` = ids con entrada B (839 en la corrida completa).

In [3]:
FUENTES = ad.cargar_fuentes_104()
PAPERS = ad.cargar_papers()
CTX = FUENTES.contextos

CONJ = {"gold_v2": eb.casos_para_chunks([{"context_id": c} for c in IDS], CTX)}
CASOS_CHUNKS, EXCL_CHUNKS = servicio.construir_casos_split(CONJ, CTX, PAPERS, ad.CACHE_OCL, ad.integrar, ad.fragmentar)
TOP3 = {"gold_v2": servicio.agrupar_top3(cli.leer_csv_top3(EV_B / "top3_gold_v2.csv"))}
INTER = servicio.interseccion(CONJ, CASOS_CHUNKS, EXCL_CHUNKS, TOP3, PAPERS)
ENTRADAS_B = eb.entradas_por_contexto(INTER["entradas"])
COB = {c for c in GOLD if c in ENTRADAS_B}
IDS_COB = [c for c in IDS if c in ENTRADAS_B]

MAN_B = json.loads((EV_B / "manifiesto_gold_b.json").read_text(encoding="utf-8"))
if not DRY:
    assert len(IDS_COB) == MAN_B["gold_v2_cobertura"] == 839, (len(IDS_COB), MAN_B["gold_v2_cobertura"])
    assert dominio.sha_ids(IDS_COB) == MAN_B["ids_gold_v2_cobertura_sha256"], "La cobertura B del Gold cambió"
print("cobertura B:", len(IDS_COB), "de", len(IDS), "| evaluables en COB:", len(COB))

SUBCONJUNTOS = {"todo": set(GOLD), "humano": HUMANO, "cob": COB}
for fuente in sorted(set(FUENTE.values())):
    SUBCONJUNTOS[f"fuente_{fuente}"] = {c for c in GOLD if FUENTE[c] == fuente}

cobertura B: 839 de 1533 | evaluables en COB: 839


In [4]:
# Fugas: el gold no puede compartir id, citante, par ni texto con los datos de entrenamiento de #104 ni de A4
def casos_fuga(filas):
    return [{"context_id": x["context_id"], "citing_id": CTX[x["context_id"]]["citing_id"],
             "cited_id": CTX[x["context_id"]]["refid"], "citation_context": CTX[x["context_id"]]["masked_text"]}
            for x in filas if x["context_id"] in CTX]


GOLD_FUGA = casos_fuga([{"context_id": c} for c in IDS_TODO])
FUGAS = {}
fuentes_entrenamiento = {p.stem: p for p in sorted((ART / "experimentos_104/datos").glob("*.csv"))}
fuentes_entrenamiento["a4_train"] = ART / "clasificacion_a4/preparacion/etiquetas_train.csv"
for nombre, ruta in fuentes_entrenamiento.items():
    filas = list({x["context_id"]: x for x in ad.leer_csv(ruta)}.values())
    r = dominio.verificar_fugas({"entrenamiento": casos_fuga(filas), "gold": GOLD_FUGA})   # lanza ValueError si hay fuga
    FUGAS[nombre] = {"casos": len(filas), "total_fugas": r["total_fugas"]}
print(pd.DataFrame(FUGAS).T.to_string())
assert sum(v["total_fugas"] for v in FUGAS.values()) == 0

                   casos  total_fugas
haiku_solo          2180            0
hibrido            13719            0
jurado              8887            0
natural            13331            0
qlora_p50          13332            0
qlora_p80           9826            0
sonnet             13325            0
sonnet_alta         9066            0
sonnet_cap         10638            0
validacion          3379            0
validacion_sonnet   3379            0
a4_train            2245            0


## Registro de modelos

Cada modelo se guarda con sus predicciones `{context_id: pred}` y los archivos de origen. La cobertura de un modelo es el conjunto de ids
que predijo: A sobre todo el gold; B / AB / B_barajada / B_sin_seccion solo sobre `COB`.

In [5]:
MODELOS = []


def registrar(nombre, familia, encoder, entrada, pista, semilla, preds, archivos, run_id_origen="", checkpoint_sha256=""):
    MODELOS.append({"nombre": nombre, "familia": familia, "encoder": encoder, "entrada": entrada, "pista": pista,
                    "semilla": semilla, "preds": preds, "archivos": archivos, "run_id_origen": run_id_origen,
                    "checkpoint_sha256": checkpoint_sha256})


def leer_preds(ruta: Path) -> dict:
    return {r["context_id"]: r["pred"] for r in map(json.loads, (l for l in ruta.read_text(encoding="utf-8").splitlines() if l.strip()))}


def n_lineas(ruta: Path) -> int:
    return sum(1 for l in ruta.read_text(encoding="utf-8").splitlines() if l.strip())


def inferir(nombre: str, etiqueta: str, casos: list, generar):
    # Escribe pred_test_gold_<etiqueta>.jsonl (salta si ya existe con n correcto) y devuelve (preds, info del archivo)
    ruta = PRED_DIR / nombre / f"pred_test_gold_{etiqueta}.jsonl"
    if not (ruta.is_file() and n_lineas(ruta) == len(casos)):
        ex.escribir_predicciones_ciegas(ruta, generar())
    return leer_preds(ruta), {"n": len(casos), "sha256": sd.sha_archivo(ruta), "ruta": str(ruta.relative_to(REPO))}

## 3. #104-b v2 (sin inferencia)

Se leen las 36 corridas de `manifiesto_gold_b.json` verificando el sha256 de cada archivo de predicciones. En DRY se recortan a los ids usados.

In [6]:
ids_uso = set(IDS)
for nombre, r in MAN_B["runs"].items():
    try:
        preds, archivos = {}, {}
        for etiqueta, a in r["archivos"].items():
            ruta = REPO / a["ruta"]
            assert sd.sha_archivo(ruta) == a["sha256"], f"sha256 distinto en {a['ruta']}"
            preds |= {c: p for c, p in leer_preds(ruta).items() if c in ids_uso}
            archivos[etiqueta] = {"n": a["n"], "sha256": a["sha256"], "ruta": a["ruta"]}
        registrar(nombre, "104b_v2", r["encoder"], r["entrada"], r["pista"], r["semilla"], preds, archivos,
                  r["run_id"], r["checkpoint_sha256"])
    except Exception:
        FALLAS.append(f"104b-{nombre}")
        print(f"[FALLO] {nombre}\n{traceback.format_exc()}")
print("#104-b v2:", sum(m["familia"] == "104b_v2" for m in MODELOS), "de", len(MAN_B["runs"]))

#104-b v2: 36 de 36


## 4. Ganadores de #104 (inferencia)

SciBERT A-natural-kd y S-sonnet (lr 3e-5, 4 épocas, semillas 42/1234/2026) y SPECTER A-natural-kd / S-sonnet (semilla 42). Entrada A sobre todo el gold.

In [7]:
GANADORES = [(enc, "M" if "natural" in base else "S", s, f"{enc}-{base}-s{s}")
             for enc, base, semillas in (("scibert", "A-natural-kd-lr3e-05-e4", (42, 1234, 2026)),
                                          ("scibert", "S-sonnet-lr3e-05-e4", (42, 1234, 2026)),
                                          ("specter", "A-natural-kd-lr3e-05-e4", (42,)),
                                          ("specter", "S-sonnet-lr3e-05-e4", (42,)))
             for s in semillas]
CASOS_A = ex.casos_gold(IDS, FUENTES, PAPERS, "A")
print("casos A:", len(CASOS_A), "de", len(IDS))
for enc, pista, s, nombre in GANADORES:
    carpeta = CORRIDAS_104 / nombre
    try:
        man = json.loads((carpeta / "manifiesto_entrenamiento.json").read_text(encoding="utf-8"))
        preds, info = inferir(nombre, "todo", CASOS_A, lambda: ex.predecir_gold(carpeta, enc, CASOS_A, "A"))
        registrar(nombre, "104_ganador", enc, "A", pista, s, preds, {"todo": info}, nombre, man["checkpoint_sha256"])
        print(f"[ok] {nombre}: {len(preds)} predicciones")
    except Exception:
        FALLAS.append(f"104-{nombre}")
        print(f"[FALLO] {nombre}\n{traceback.format_exc()}")
    finally:
        liberar_gpu()

casos A: 1533 de 1533
[ok] scibert-A-natural-kd-lr3e-05-e4-s42: 1533 predicciones
[ok] scibert-A-natural-kd-lr3e-05-e4-s1234: 1533 predicciones
[ok] scibert-A-natural-kd-lr3e-05-e4-s2026: 1533 predicciones
[ok] scibert-S-sonnet-lr3e-05-e4-s42: 1533 predicciones
[ok] scibert-S-sonnet-lr3e-05-e4-s1234: 1533 predicciones
[ok] scibert-S-sonnet-lr3e-05-e4-s2026: 1533 predicciones
[ok] specter-A-natural-kd-lr3e-05-e4-s42: 1533 predicciones
[ok] specter-S-sonnet-lr3e-05-e4-s42: 1533 predicciones


## 5. TF-IDF + regresión logística

Se reajusta con la misma configuración de `ex.linea_base_tfidf` sobre `natural.csv` y `sonnet.csv` (entrada A) y se predice el gold.

In [8]:
def texto(c):
    return f"{c['citation_context']} {c['titulo']} {c['abstract']}"


for pista, csv in (("M", "natural"), ("S", "sonnet")):
    nombre = f"tfidf-{csv}"
    try:
        filas = ad.leer_csv(ART / f"experimentos_104/datos/{csv}.csv")
        entrenamiento = [c for c in ex.casos_a(filas, FUENTES, PAPERS)[0] if c["indice"] is not None]
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200_000)
        clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(
            vec.fit_transform([texto(c) for c in entrenamiento]), [c["indice"] for c in entrenamiento])

        def generar():
            probs = clf.predict_proba(vec.transform([texto(c) for c in CASOS_A]))
            return [{"context_id": c["context_id"], "pred": int(clf.classes_[p.argmax()]), "probs": [float(x) for x in p]}
                    for c, p in zip(CASOS_A, probs)]

        ruta = PRED_DIR / nombre / "pred_test_gold_todo.jsonl"
        ex.escribir_predicciones_ciegas(ruta, generar())   # el ajuste es barato: se recalcula siempre
        info = {"n": len(CASOS_A), "sha256": sd.sha_archivo(ruta), "ruta": str(ruta.relative_to(REPO))}
        registrar(nombre, "tfidf", "tfidf", "A", pista, 0, leer_preds(ruta), {"todo": info}, nombre, "")
        print(f"[ok] {nombre}: entrenó con {len(entrenamiento)} casos")
    except Exception:
        FALLAS.append(f"tfidf-{csv}")
        print(f"[FALLO] {nombre}\n{traceback.format_exc()}")

[ok] tfidf-natural: entrenó con 13331 casos
[ok] tfidf-sonnet: entrenó con 13325 casos


## 6. A4 (18 corridas)

A usa todo el gold; B / B_sin_seccion / B_barajada, solo `COB` (mismo camino que `evaluar_conjunto_extra`, sin etiquetas). Antes se comprueba que
la especificación de entrada coincida con `input_spec_sha256` del manifiesto; si no, la corrida se omite y queda en `FALLAS`.

In [9]:
FILAS_COB = [{"context_id": c, "citing_id": CTX[c]["citing_id"]} for c in IDS_COB]
ESPECIFICACION_OK = {}


def predecir_a4(carpeta, encoder, entrada, semilla, filas):
    _, hp, tok, dev, modelo = ex._cargar_modelo(carpeta, encoder)
    casos, _ = ex.casos_entrada(entrada, filas, FUENTES, PAPERS, ENTRADAS_B, etiqueta=None)
    ej = ct.preparar_ejemplos(ex._limpio(ex._barajar(entrada, casos, semilla)), entrada, tok, hp.max_len)
    pred, probs, _ = ex._predicciones(modelo, ej, tok.pad_id, dev)
    return [{"context_id": e["context_id"], "pred": p, "probs": pr} for e, p, pr in zip(ej, pred, probs)]


for carpeta in sorted(p for p in CORRIDAS_A4.iterdir() if p.is_dir()):
    man = json.loads((carpeta / "manifiesto_entrenamiento.json").read_text(encoding="utf-8"))
    enc, entrada, s = man["encoder"], man["entrada"], man["semilla"]
    nombre = f"a4-{man['experiment_id']}"
    try:
        esperado = dominio.sha_especificacion(dominio.especificacion_entrada(
            entrada, 512, s if entrada == "B_barajada" else None))
        ESPECIFICACION_OK[nombre] = esperado == man["input_spec_sha256"]
        if not ESPECIFICACION_OK[nombre]:
            FALLAS.append(f"spec-{nombre}")
            print(f"[omitido] {nombre}: input_spec_sha256 no coincide")
            continue
        if entrada == "A":
            etiqueta, casos = "todo", CASOS_A
            generar = lambda: ex.predecir_gold(carpeta, enc, CASOS_A, "A")
        else:
            etiqueta = "cob"
            casos = ex.casos_entrada(entrada, FILAS_COB, FUENTES, PAPERS, ENTRADAS_B, etiqueta=None)[0]
            generar = lambda: predecir_a4(carpeta, enc, entrada, s, FILAS_COB)
        preds, info = inferir(nombre, etiqueta, casos, generar)
        registrar(nombre, "a4", enc, entrada, "A4", s, preds, {etiqueta: info}, man["experiment_id"], man["checkpoint_sha256"])
        print(f"[ok] {nombre}: {len(preds)} predicciones")
    except Exception:
        FALLAS.append(f"a4-{nombre}")
        print(f"[FALLO] {nombre}\n{traceback.format_exc()}")
    finally:
        liberar_gpu()
print("modelos registrados:", len(MODELOS), "| fallas:", FALLAS or "ninguna", f"| {(time.time() - T0) / 60:.1f} min")

[ok] a4-scibert_A_1234: 1533 predicciones
[ok] a4-scibert_A_2026: 1533 predicciones
[ok] a4-scibert_A_42: 1533 predicciones
[ok] a4-scibert_B_1234: 839 predicciones
[ok] a4-scibert_B_2026: 839 predicciones
[ok] a4-scibert_B_42: 839 predicciones
[ok] a4-scibert_B_barajada_1234: 839 predicciones
[ok] a4-scibert_B_barajada_2026: 839 predicciones
[ok] a4-scibert_B_barajada_42: 839 predicciones
[ok] a4-scibert_B_sin_seccion_1234: 839 predicciones
[ok] a4-scibert_B_sin_seccion_2026: 839 predicciones
[ok] a4-scibert_B_sin_seccion_42: 839 predicciones
[ok] a4-specter_A_1234: 1533 predicciones
[ok] a4-specter_A_2026: 1533 predicciones
[ok] a4-specter_A_42: 1533 predicciones
[ok] a4-specter_B_1234: 839 predicciones
[ok] a4-specter_B_2026: 839 predicciones
[ok] a4-specter_B_42: 839 predicciones
modelos registrados: 64 | fallas: ninguna | 1.1 min


## 7. Métricas

`dominio.metricas(verdad, pred)` por modelo × subconjunto (`todo`, `humano`, `cob` y por `fuente`). Un modelo se mide en un subconjunto solo si predijo
todos sus ids (A sobre el gold completo; los modelos con chunks solo en `cob`).

In [10]:
FILAS, POR_CLASE, METRICAS = [], [], {}
for m in MODELOS:
    for sub, ids_sub in SUBCONJUNTOS.items():
        ids_m = sorted(ids_sub)
        if not ids_m or any(c not in m["preds"] for c in ids_m):
            continue
        r = dominio.metricas([GOLD[c] for c in ids_m], [m["preds"][c] for c in ids_m])
        METRICAS[(m["nombre"], sub)] = r
        FILAS.append({"nombre": m["nombre"], "familia": m["familia"], "encoder": m["encoder"], "entrada": m["entrada"],
                      "pista": m["pista"], "semilla": m["semilla"], "subconjunto": sub, "n": r["n"],
                      "accuracy": r["accuracy"], "f1_macro": r["f1_macro"], "f1_micro": r["f1_micro"], "f1_weighted": r["f1_weighted"]})
        POR_CLASE += [{"nombre": m["nombre"], "familia": m["familia"], "subconjunto": sub, "clase": c["label"],
                       "precision": c["precision"], "recall": c["recall"], "f1": c["f1"], "soporte": c["support"]}
                      for c in r["per_class"]]
TABLA = pd.DataFrame(FILAS)
POR_CLASE_DF = pd.DataFrame(POR_CLASE)
CONFIG = ["familia", "encoder", "entrada", "pista", "subconjunto"]
AGREGADO = (TABLA.groupby(CONFIG)
            .agg(n_semillas=("semilla", "nunique"), n=("n", "first"),
                 **{f"{k}_media": (k, "mean") for k in ("accuracy", "f1_macro", "f1_micro", "f1_weighted")},
                 **{f"{k}_sd": (k, "std") for k in ("accuracy", "f1_macro", "f1_micro", "f1_weighted")})
            .reset_index())
for sub in ("todo", "humano", "cob"):
    print(f"=== {sub}")
    print(AGREGADO[AGREGADO["subconjunto"] == sub].drop(columns="subconjunto")
          .sort_values("f1_macro_media", ascending=False).round(4).to_string(index=False))

=== todo
    familia encoder entrada pista  n_semillas    n  accuracy_media  f1_macro_media  f1_micro_media  f1_weighted_media  accuracy_sd  f1_macro_sd  f1_micro_sd  f1_weighted_sd
104_ganador scibert       A     S           3 1531          0.7213          0.5995          0.7213             0.7086       0.0062       0.0096       0.0062          0.0085
104_ganador specter       A     S           1 1531          0.7067          0.5821          0.7067             0.6935          NaN          NaN          NaN             NaN
    104b_v2 scibert       A     S           5 1531          0.7033          0.5790          0.7033             0.6888       0.0024       0.0031       0.0024          0.0027
    104b_v2 specter       A     S           1 1531          0.6845          0.5616          0.6845             0.6697          NaN          NaN          NaN             NaN
104_ganador scibert       A     M           3 1531          0.6791          0.5562          0.6791             0.6683       0.

## 8. Comparaciones

F1 por clase de la mejor configuración de cada familia (mejor macro-F1 medio en `cob`), matriz de confusión del mejor modelo y contrastes pareados
**AB − A** y **B − A** en `COB` para #104-b v2 (pistas M y S; media de semillas, `prueba_combinada`) con corrección de Holm.

In [11]:
COB_AGG = AGREGADO[AGREGADO["subconjunto"] == "cob"]
MEJOR_POR_FAMILIA = COB_AGG.loc[COB_AGG.groupby("familia")["f1_macro_media"].idxmax()]
filas_clase = []
for r in MEJOR_POR_FAMILIA.itertuples():
    nombres = TABLA[(TABLA.familia == r.familia) & (TABLA.encoder == r.encoder) & (TABLA.entrada == r.entrada)
                    & (TABLA.pista == r.pista) & (TABLA.subconjunto == "cob")]["nombre"]
    sel = POR_CLASE_DF[POR_CLASE_DF["nombre"].isin(nombres) & (POR_CLASE_DF["subconjunto"] == "cob")]
    media = sel.groupby("clase", sort=False)["f1"].mean()
    filas_clase.append(media.rename(f"{r.familia}:{r.encoder}-{r.entrada}-{r.pista}"))
F1_CLASE = pd.concat(filas_clase, axis=1)
F1_CLASE["soporte"] = pd.Series({c["label"]: c["support"] for c in METRICAS[(nombres.iloc[0], "cob")]["per_class"]})
print(F1_CLASE.round(3).to_string())

MEJOR_MODELO = TABLA[TABLA.subconjunto == "cob"].sort_values("f1_macro", ascending=False).iloc[0]["nombre"]
CONF = METRICAS[(MEJOR_MODELO, "cob")]["confusion_matrix"]
print("\nMejor modelo en cob:", MEJOR_MODELO)
print(pd.DataFrame(CONF, index=[c[:12] for c in dominio.CATEGORIAS], columns=[c[:6] for c in dominio.CATEGORIAS]).to_string())

                                  104_ganador:scibert-A-S  104b_v2:scibert-AB-S  a4:scibert-B_barajada-A4  tfidf:tfidf-A-S  soporte
clase                                                                                                                              
Background                                          0.761                 0.735                     0.687            0.517      238
Gap                                                 0.518                 0.477                     0.351            0.299       66
Basis                                               0.362                 0.464                     0.130            0.135       22
Comparison                                          0.790                 0.771                     0.651            0.540      159
Application                                         0.846                 0.831                     0.810            0.561      178
Improvement / Modification                          0.275                 0.

In [12]:
POR_CLAVE = {(m["familia"], m["encoder"], m["entrada"], m["pista"], m["semilla"]): m for m in MODELOS}
IDS_COB_EVAL = sorted(COB)
ETIQ = [GOLD[c] for c in IDS_COB_EVAL]
GRUPOS = [CTX[c]["citing_id"] for c in IDS_COB_EVAL]


def semillas_de(familia, encoder, entrada, pista):
    return {k[4] for k in POR_CLAVE if k[:4] == (familia, encoder, entrada, pista)
            and all(c in POR_CLAVE[k]["preds"] for c in IDS_COB_EVAL)}


def contraste(familia, encoder, x, a, pista):
    seeds = sorted(semillas_de(familia, encoder, x, pista) & semilla_a if (semilla_a := semillas_de(familia, encoder, a, pista)) else [])
    if len(seeds) < 2:
        return None
    preds = lambda ent: [[POR_CLAVE[(familia, encoder, ent, pista, s)]["preds"][c] for c in IDS_COB_EVAL] for s in seeds]
    r = ex.prueba_combinada(preds(x), preds(a), ETIQ, GRUPOS, n=N_BOOTSTRAP, semilla=SEMILLA_BOOTSTRAP)
    return {**r, "semillas": seeds, "n": len(IDS_COB_EVAL)}


CONTRASTES = {}
for pista in ("M", "S"):
    for x in ("AB", "B"):
        try:
            if (r := contraste("104b_v2", "scibert", x, "A", pista)) is not None:
                CONTRASTES[f"{pista}:{x} - A"] = r
        except Exception:
            FALLAS.append(f"contraste-{pista}-{x}")
            print(f"[FALLO] contraste {pista} {x}-A\n{traceback.format_exc()}")
if CONTRASTES:
    p_holm = ex.holm({k: v["p"] for k, v in CONTRASTES.items()})
    for k, v in CONTRASTES.items():
        v["p_holm"] = p_holm[k]
print(pd.DataFrame({k: {c: v.get(c) for c in ("diferencia", "se", "gl", "p", "p_holm")} for k, v in CONTRASTES.items()}).T.to_string())
print({k: v for k, v in next(iter(CONTRASTES.values())).items() if k not in ("por_clase",)} if CONTRASTES else "sin contrastes")

          diferencia        se         gl         p  p_holm
M:AB - A   -0.009364  0.009252   9.828679  0.335760     1.0
M:B - A    -0.008409  0.011195  10.266382  0.469443     1.0
S:AB - A    0.015151  0.014338  20.523375  0.302938     1.0
S:B - A     0.000986  0.011474  34.640403  0.931980     1.0
{'metrica': 'f1_macro', 'diferencia': -0.009363954247728335, 'diferencia_por_semilla': [-0.03150057661484007, -0.009965399850025425, 0.0023657756858727064, -0.018065415057564316, 0.010345844597915432], 'sd_semillas': 0.01652399725938538, 'v_sem': 5.460849708563511e-05, 'v_art': 3.099229837058139e-05, 'se': 0.009252069793090435, 'gl': 9.828678826225401, 'p': 0.33575997159216797, 'ic95': (-0.03002766142322251, 0.01129975292776584), 'delta': 0.02, 'p_tost_inferior': 0.138759398934307, 'p_tost_superior': 0.005067092284601491, 'p_tost': 0.138759398934307, 'ic90': (-0.02616268009946449, 0.007434771604007819), 'semillas': [7, 42, 1234, 2026, 31337], 'casos': 839, 'grupos': 522, 'n_remuestras': 5000

## 9. MLflow

Un run por modelo evaluado en el experimento `evaluacion-test-gold` (tags `run_id_origen`, `checkpoint_sha256`, `gold_sha256`). En DRY usa una BD aparte.
La BD real fue migrada con MLflow 3.18: si el entorno no puede abrirla, el paso falla sin afectar a las evidencias del paso 10.

In [13]:
SUBS_MLFLOW = {"todo": "test_gold", "humano": "test_gold_humano", "cob": "test_gold_cob"}
try:
    if DRY:
        rm.configurar("evaluacion-test-gold" + SUFIJO.replace("_", "-"), db_path=ART / "test_gold_dry/mlflow_dry.db",
                      artifacts_dir=ART / "test_gold_dry/mlartifacts")
    else:
        rm.configurar("evaluacion-test-gold")
    for m in MODELOS:
        with mlflow.start_run(run_name=m["nombre"]):
            mlflow.set_tags({"run_id_origen": m["run_id_origen"], "checkpoint_sha256": m["checkpoint_sha256"],
                             "gold_sha256": GOLD_SHA, "familia": m["familia"], "encoder": m["encoder"],
                             "entrada": m["entrada"], "pista": m["pista"], "semilla": str(m["semilla"])})
            for sub, prefijo in SUBS_MLFLOW.items():
                if (m["nombre"], sub) in METRICAS:
                    rm.log_metricas(prefijo, METRICAS[(m["nombre"], sub)])
                    rm.log_matriz_confusion(METRICAS[(m["nombre"], sub)], prefijo)
    print("MLflow:", mlflow.get_tracking_uri(), "|", len(MODELOS), "runs")
except Exception:
    FALLAS.append("mlflow")
    print(f"[FALLO] MLflow\n{traceback.format_exc()}")

MLflow: sqlite:////home/carlos/microproyecto-local-citation/mlflow.db | 64 runs


## 10. Evidencias

In [14]:
TABLA.to_csv(EVIDENCIAS / "metricas_test_gold.csv", index=False)
AGREGADO.to_csv(EVIDENCIAS / "metricas_agregadas.csv", index=False)
POR_CLASE_DF.to_csv(EVIDENCIAS / "metricas_por_clase.csv", index=False)
escribir_json("contrastes.json", {"n_bootstrap": N_BOOTSTRAP, "semilla": SEMILLA_BOOTSTRAP, "subconjunto": "cob",
                                  "mejor_modelo_cob": MEJOR_MODELO, "matriz_confusion_mejor": CONF, "contrastes": CONTRASTES})
escribir_json("manifiesto_test_gold.json", {
    "dry": DRY, "gold_csv": str(GOLD_CSV.relative_to(REPO)), "gold_sha256": GOLD_SHA, "n_gold": len(IDS_TODO), "n_evaluables": len(GOLD),
    "n_cob": len(COB), "ids_cob_sha256": dominio.sha_ids(IDS_COB), "fugas": FUGAS, "fallas": FALLAS,
    "especificacion_a4_ok": ESPECIFICACION_OK,
    "modelos": {m["nombre"]: {"familia": m["familia"], "encoder": m["encoder"], "entrada": m["entrada"], "pista": m["pista"],
                              "semilla": m["semilla"], "run_id_origen": m["run_id_origen"],
                              "checkpoint_sha256": m["checkpoint_sha256"], "archivos": m["archivos"]} for m in MODELOS}})
print("evidencias en", EVIDENCIAS.relative_to(REPO), "| fallas:", FALLAS or "ninguna")

evidencias en proyecto_de_grado/docs/evidencias_test_gold | fallas: ninguna


## Sanidad

1. El macro-F1 de una corrida #104-b v2 recalculado desde su archivo de predicciones coincide con la tabla.
2. Una corrida A4-A reinferida dos veces da las mismas predicciones.

In [15]:
m104 = next(m for m in MODELOS if m["familia"] == "104b_v2" and "todo" in m["archivos"])
recargado = leer_preds(REPO / m104["archivos"]["todo"]["ruta"])
ids_m = sorted(SUBCONJUNTOS["todo"])
f1 = dominio.metricas([GOLD[c] for c in ids_m], [recargado[c] for c in ids_m])["f1_macro"]
assert abs(f1 - METRICAS[(m104["nombre"], "todo")]["f1_macro"]) < 1e-12
print("recarga #104-b:", m104["nombre"], round(f1, 4))

a4a = next(p for p in sorted(CORRIDAS_A4.iterdir()) if p.name.startswith("scibert_A_"))
dos = [[(r["context_id"], r["pred"]) for r in ex.predecir_gold(a4a, "scibert", CASOS_A[:64], "A")] for _ in range(2)]
assert dos[0] == dos[1]
print("A4-A reinferida dos veces: iguales", a4a.name)

recarga #104-b: scibert-A-M-lr2e-05-s42 0.5421


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 36881.56it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

A4-A reinferida dos veces: iguales scibert_A_1234
